# MegaStore Commercial Performance & Profitability Analysis

## Project Objective

The purpose of this project is to evaluate MegaStore's commercial performance and identify where the business is generating profitable growth and where revenue is being weakened by discounts, product performance, shipping costs, and regional differences.

Rather than focusing only on total sales, this analysis will investigate the underlying economics of the business to understand whether revenue is translating into sustainable profit.

Using Python, SQL, and Power BI, the project will explore:

- Profit leakage across products, categories, markets, and regions
- The relationship between discounting, sales, and profitability
- Product portfolio performance and products that generate high revenue but weak or negative profit
- The impact of shipping costs and fulfilment decisions on profitability
- Geographic differences in revenue, profit, and profit margins
- Order-level factors associated with strong and weak commercial performance
- Whether sales growth over time is accompanied by healthy profit growth

## Business Question

**Where is MegaStore generating profitable growth, where is it losing margin, and what commercial and operational actions could improve overall profitability?**

## Tools

- **Python** — data cleaning, validation, exploratory analysis, and commercial diagnostics
- **SQL** — business-focused querying and profitability analysis
- **Power BI** — executive dashboard development and presentation of actionable insights

## Expected Outcome

The final analysis will provide evidence-based recommendations that could help management improve pricing and discount decisions, product portfolio performance, shipping efficiency, regional profitability, and the overall quality of business growth.

In [1]:
# import tools:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/megastore_sales.csv")

df.head()

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
0,AG-2011-2040,1/1/2011,6/1/2011,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,...,Office Supplies,Storage,"Tenex Lockers, Blue",408,2,0.0,106.140,35.46,Medium,2011
1,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Office Supplies,Supplies,"Acme Trimmer, High Speed",120,3,0.1,36.036,9.72,Medium,2011
2,HU-2011-1220,1/1/2011,5/1/2011,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,...,Office Supplies,Storage,"Tenex Box, Single Width",66,4,0.0,29.640,8.17,High,2011
3,IT-2011-3647632,1/1/2011,5/1/2011,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,...,Office Supplies,Paper,"Enermax Note Cards, Premium",45,3,0.5,-26.055,4.82,High,2011
4,IN-2011-47883,1/1/2011,8/1/2011,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,...,Furniture,Furnishings,"Eldon Light Bulb, Duo Pack",114,5,0.1,37.770,4.70,Medium,2011


In [2]:
df.tail()

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
51285,CA-2014-115427,31-12-2014,4/1/2015,Standard Class,Erica Bern,Corporate,California,United States,US,West,...,Office Supplies,Binders,"Cardinal Slant-D Ring Binder, Heavy Gauge Vinyl",14,2,0.2,4.5188,0.89,Medium,2014
51286,MO-2014-2560,31-12-2014,5/1/2015,Standard Class,Liz Preis,Consumer,Souss-Massa-Draâ,Morocco,Africa,Africa,...,Office Supplies,Binders,"Wilson Jones Hole Reinforcements, Clear",4,1,0.0,0.4200,0.49,Medium,2014
51287,MX-2014-110527,31-12-2014,2/1/2015,Second Class,Charlotte Melton,Consumer,Managua,Nicaragua,LATAM,Central,...,Office Supplies,Labels,"Hon Color Coded Labels, 5000 Label Set",26,3,0.0,12.3600,0.35,Medium,2014
51288,MX-2014-114783,31-12-2014,6/1/2015,Standard Class,Tamara Dahlen,Consumer,Chihuahua,Mexico,LATAM,North,...,Office Supplies,Labels,"Hon Legal Exhibit Labels, Alphabetical",7,1,0.0,0.5600,0.20,Medium,2014
51289,CA-2014-156720,31-12-2014,4/1/2015,Standard Class,Jill Matthias,Consumer,Colorado,United States,US,West,...,Office Supplies,Fasteners,Bagged Rubber Bands,3,3,0.2,-0.6048,0.17,Medium,2014


## 2. Initial Data Inspection

Before cleaning or analysing the dataset, an initial inspection is performed to understand its structure, data types, completeness, and potential data-quality issues.

In [3]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Dataset shape: (51290, 21)

Column names:
['order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_name', 'segment', 'state', 'country', 'market', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit', 'shipping_cost', 'order_priority', 'year']

Data types:
order_id           object
order_date         object
ship_date          object
ship_mode          object
customer_name      object
segment            object
state              object
country            object
market             object
region             object
product_id         object
category           object
sub_category       object
product_name       object
sales              object
quantity            int64
discount          float64
profit            float64
shipping_cost     float64
order_priority     object
year                int64
dtype: object

Missing values:
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_name     0
seg

## 3. Data Quality Assessment

The initial inspection identified three columns requiring further investigation before analysis:

- `order_date` and `ship_date` are stored as text rather than datetime values.
- `sales` is stored as text rather than a numeric value.

These columns will be examined before conversion to ensure that formatting issues are handled correctly and that no information is lost during data cleaning.

In [4]:
print("Sample order dates:")
print(df["order_date"].head(10).tolist())

print("\nSample ship dates:")
print(df["ship_date"].head(10).tolist())

print("\nSample sales values:")
print(df["sales"].head(20).tolist())

print("\nSales values containing commas:")
print(df.loc[df["sales"].str.contains(",", na=False), "sales"].head(10).tolist())

Sample order dates:
['1/1/2011', '1/1/2011', '1/1/2011', '1/1/2011', '1/1/2011', '1/1/2011', '2/1/2011', '3/1/2011', '3/1/2011', '3/1/2011']

Sample ship dates:
['6/1/2011', '8/1/2011', '5/1/2011', '5/1/2011', '8/1/2011', '8/1/2011', '6/1/2011', '3/1/2011', '9/1/2011', '7/1/2011']

Sample sales values:
['408', '120', '66', '45', '114', '55', '314', '276', '912', '667', '338', '211', '854', '193', '159', '195', '123', '69', '69', '135']

Sales values containing commas:
['1,648', '3,029', '2,574', '2,479', '2,174', '2,875', '1,384', '1,323', '1,200', '1,326']


## 4. Data Cleaning and Type Conversion

Following the data-quality assessment, the following transformations are applied:

- Convert `order_date` and `ship_date` from text to datetime using day-first date interpretation.
- Remove thousands separators from `sales` and convert the column to numeric format.
- Validate the conversions to ensure that no values are lost or incorrectly converted.

In [5]:
# Convert date columns
df["order_date"] = pd.to_datetime(
    df["order_date"],
    dayfirst=True,
    errors="coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    dayfirst=True,
    errors="coerce"
)

# Remove thousands separators and convert sales to numeric
df["sales"] = (
    df["sales"]
    .str.replace(",", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

In [6]:
print("Updated data types:")
print(df[["order_date", "ship_date", "sales"]].dtypes)

print("\nMissing values after conversion:")
print(df[["order_date", "ship_date", "sales"]].isnull().sum())

print("\nDate range:")
print("Order date:", df["order_date"].min(), "to", df["order_date"].max())
print("Ship date:", df["ship_date"].min(), "to", df["ship_date"].max())

print("\nSales range:")
print("Minimum sales:", df["sales"].min())
print("Maximum sales:", df["sales"].max())

Updated data types:
order_date    datetime64[ns]
ship_date     datetime64[ns]
sales                  int64
dtype: object

Missing values after conversion:
order_date    31223
ship_date     31456
sales             0
dtype: int64

Date range:
Order date: 2011-01-01 00:00:00 to 2014-12-12 00:00:00
Ship date: 2011-01-03 00:00:00 to 2015-01-07 00:00:00

Sales range:
Minimum sales: 0
Maximum sales: 22638


In [7]:
df = pd.read_csv("../data/megastore_sales.csv")

In [8]:
# conver sales

df["sales"] = (
    df["sales"]
    .str.replace(",", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

In [9]:
# For the dates, use pandas' mixed-format parser:

df["order_date"] = pd.to_datetime(
    df["order_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

df["ship_date"] = pd.to_datetime(
    df["ship_date"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

In [11]:
print("Updated data types:")
print(df[["order_date", "ship_date", "sales"]].dtypes)

print("\nMissing values after conversion:")
print(df[["order_date", "ship_date", "sales"]].isnull().sum())

print("\nDate range:")
print("Order date:", df["order_date"].min(), "to", df["order_date"].max())
print("Ship date:", df["ship_date"].min(), "to", df["ship_date"].max())

print("\nSales range:")
print("Minimum sales:", df["sales"].min())
print("Maximum sales:", df["sales"].max())

Updated data types:
order_date    datetime64[ns]
ship_date     datetime64[ns]
sales                  int64
dtype: object

Missing values after conversion:
order_date    0
ship_date     0
sales         0
dtype: int64

Date range:
Order date: 2011-01-01 00:00:00 to 2014-12-31 00:00:00
Ship date: 2011-01-03 00:00:00 to 2015-01-07 00:00:00

Sales range:
Minimum sales: 0
Maximum sales: 22638


## 5. Business Data Validation

After correcting the data types, the dataset is validated from a business perspective.

The purpose of this stage is to identify unusual or potentially problematic transactions that could affect the profitability analysis. Checks include zero or negative sales, quantities, discounts, profits, shipping costs, and the relationship between order and shipping dates.

Potential anomalies are investigated rather than automatically removed, because unusual transactions may represent genuine business activity and could themselves provide useful commercial insights.

In [12]:
print("Zero sales:", (df["sales"] == 0).sum())
print("Negative sales:", (df["sales"] < 0).sum())

print("\nZero quantity:", (df["quantity"] == 0).sum())
print("Negative quantity:", (df["quantity"] < 0).sum())

print("\nNegative profit:", (df["profit"] < 0).sum())
print("Zero profit:", (df["profit"] == 0).sum())

print("\nZero shipping cost:", (df["shipping_cost"] == 0).sum())
print("Negative shipping cost:", (df["shipping_cost"] < 0).sum())

print("\nDiscount range:")
print("Minimum:", df["discount"].min())
print("Maximum:", df["discount"].max())

print("\nOrders shipped before order date:")
print((df["ship_date"] < df["order_date"]).sum())

Zero sales: 1
Negative sales: 0

Zero quantity: 0
Negative quantity: 0

Negative profit: 12543
Zero profit: 668

Zero shipping cost: 2
Negative shipping cost: 0

Discount range:
Minimum: 0.0
Maximum: 0.85

Orders shipped before order date:
0


## 6. Investigation of Exceptional Transactions

The business validation identified a small number of zero-sales and zero-shipping-cost transactions, alongside a much larger population of loss-making transactions.

Before beginning the profitability analysis, the exceptional records are inspected individually to determine whether they represent data-quality problems or legitimate business transactions.

Loss-making transactions are retained because they are central to the project's investigation of commercial profit leakage.

In [13]:
print("Zero-sales transaction:")
display(df[df["sales"] == 0])

print("\nZero-shipping-cost transactions:")
display(df[df["shipping_cost"] == 0])

print("\nSample zero-profit transactions:")
display(df[df["profit"] == 0].head(10))

Zero-sales transaction:


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
40017,US-2014-102288,2014-06-20,2014-06-24,Standard Class,Zuschuss Carroll,Consumer,Texas,United States,US,Central,...,Office Supplies,Appliances,Hoover Replacement Belt for Commercial Guardsm...,0,1,0.8,-1.11,0.01,Medium,2014



Zero-shipping-cost transactions:


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
9827,MX-2012-140767,2012-02-18,2012-02-22,Standard Class,Ross Baird,Home Office,São Paulo,Brazil,LATAM,South,...,Office Supplies,Binders,"Acco Index Tab, Economy",13,2,0.0,2.4,0.0,Medium,2012
11846,MX-2012-134460,2012-05-22,2012-05-26,Second Class,Mick Crebagga,Consumer,Managua,Nicaragua,LATAM,Central,...,Office Supplies,Paper,"Eaton Computer Printout Paper, 8.5 x 11",61,3,0.0,1.8,0.0,High,2012



Sample zero-profit transactions:


,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,...,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year
109,HU-2011-1380,2011-01-10,2011-01-10,Same Day,Elpida Rittenbach,Corporate,Budapest,Hungary,EMEA,EMEA,...,Office Supplies,Paper,"Eaton Note Cards, Premium",53,2,0.0,0.0,16.06,Critical,2011
294,MX-2011-165526,2011-01-22,2011-01-26,Standard Class,Michael Paige,Corporate,Atlántico,Colombia,LATAM,South,...,Furniture,Chairs,"Novimex Steel Folding Chair, Red",219,4,0.0,0.0,10.23,Medium,2011
327,CA-2011-146997,2011-01-24,2011-01-28,Standard Class,Speros Goranitis,Consumer,Indiana,United States,US,Central,...,Office Supplies,Fasteners,"Alliance Big Bands Rubber Bands, 12/Pack",6,3,0.0,0.0,0.95,High,2011
370,IT-2011-2536577,2011-01-26,2011-01-29,Second Class,Alan Shonely,Consumer,Languedoc-Roussillon,France,EU,Central,...,Furniture,Furnishings,"Rubbermaid Stacking Tray, Erganomic",177,8,0.0,0.0,0.61,Medium,2011
374,US-2011-155502,2011-01-27,2011-02-01,Standard Class,Shirley Daniels,Home Office,Virginia,United States,US,South,...,Office Supplies,Storage,"SAFCO Commercial Wire Shelving, 72h",490,8,0.0,0.0,21.48,Medium,2011
378,US-2011-155502,2011-01-27,2011-02-01,Standard Class,Shirley Daniels,Home Office,Virginia,United States,US,South,...,Technology,Phones,Macally Suction Cup Mount,155,13,0.0,0.0,11.95,Medium,2011
446,ES-2011-5268439,2011-02-01,2011-02-03,Second Class,Gene Hale,Corporate,Poitou-Charentes,France,EU,Central,...,Office Supplies,Envelopes,"GlobeWeis Peel and Seal, Set of 50",21,1,0.0,0.0,3.34,High,2011
495,SF-2011-2090,2011-02-04,2011-02-09,Second Class,Tom Stivers,Corporate,Gauteng,South Africa,Africa,Africa,...,Office Supplies,Binders,"Avery Hole Reinforcements, Economy",4,1,0.0,0.0,0.19,Medium,2011
659,ES-2011-4436456,2011-02-16,2011-02-20,Standard Class,Rob Williams,Corporate,Languedoc-Roussillon,France,EU,Central,...,Office Supplies,Paper,"Enermax Parchment Paper, Premium",62,4,0.0,0.0,3.13,Medium,2011
744,ID-2011-55667,2011-02-23,2011-02-26,Second Class,Roland Fjeld,Consumer,Punjab,Pakistan,APAC,Central Asia,...,Office Supplies,Paper,"SanDisk Note Cards, Premium",45,3,0.5,0.0,1.48,High,2011


## 7. Commercial Performance Baseline

Before investigating the drivers of profit leakage, a commercial baseline is established for MegaStore.

The analysis measures overall revenue, profit, order volume, units sold, profit margin, average order value, shipping costs, and the financial impact of loss-making transactions.

These measures provide a reference point for evaluating whether growth and sales activity are translating into healthy profitability.

In [14]:
# Core commercial metrics

total_sales = df["sales"].sum()
total_profit = df["profit"].sum()
total_quantity = df["quantity"].sum()
total_orders = df["order_id"].nunique()
total_shipping_cost = df["shipping_cost"].sum()

profit_margin = (total_profit / total_sales) * 100
average_order_value = total_sales / total_orders

# Loss-making transaction lines
loss_transactions = df[df["profit"] < 0]

loss_transaction_count = len(loss_transactions)
loss_transaction_pct = (loss_transaction_count / len(df)) * 100
total_losses = loss_transactions["profit"].sum()
sales_from_loss_transactions = loss_transactions["sales"].sum()

print(f"Total Sales: £{total_sales:,.2f}")
print(f"Total Profit: £{total_profit:,.2f}")
print(f"Profit Margin: {profit_margin:.2f}%")
print(f"Total Orders: {total_orders:,}")
print(f"Units Sold: {total_quantity:,}")
print(f"Average Order Value: £{average_order_value:,.2f}")
print(f"Total Shipping Cost: £{total_shipping_cost:,.2f}")

print("\n--- Profit Leakage ---")
print(f"Loss-making transaction lines: {loss_transaction_count:,}")
print(f"Loss-making transaction rate: {loss_transaction_pct:.2f}%")
print(f"Sales from loss-making transactions: £{sales_from_loss_transactions:,.2f}")
print(f"Total losses from loss-making transactions: £{total_losses:,.2f}")

Total Sales: £12,642,905.00
Total Profit: £1,469,034.82
Profit Margin: 11.62%
Total Orders: 25,035
Units Sold: 178,312
Average Order Value: £505.01
Total Shipping Cost: £1,352,820.69

--- Profit Leakage ---
Loss-making transaction lines: 12,543
Loss-making transaction rate: 24.46%
Sales from loss-making transactions: £2,486,341.00
Total losses from loss-making transactions: £-920,357.39


## 8. Profit Leakage Analysis

Approximately 24% of MegaStore's transaction lines are loss-making, generating substantial sales while contributing more than £920,000 in losses.

This section investigates where these losses are concentrated and whether particular areas of the business contribute disproportionately to profit leakage.

The analysis begins at the product sub-category level to identify areas that combine significant commercial activity with weak profitability.

In [15]:
subcategory_profitability = (
    df.groupby("sub_category")
      .agg(
          sales=("sales", "sum"),
          profit=("profit", "sum"),
          quantity=("quantity", "sum"),
          transactions=("order_id", "size")
      )
      .reset_index()
)

subcategory_profitability["profit_margin_pct"] = (
    subcategory_profitability["profit"]
    / subcategory_profitability["sales"]
    * 100
)

subcategory_profitability = (
    subcategory_profitability
    .sort_values("profit")
)

display(
    subcategory_profitability.round({
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2
    })
)

,sub_category,sales,profit,quantity,transactions,profit_margin_pct
16,Tables,757034,-64083.39,3083,861,-8.47
8,Fasteners,83254,11525.42,8390,2420,13.84
10,Labels,73433,15010.51,9322,2606,20.44
15,Supplies,243090,22583.26,8543,2425,9.29
7,Envelopes,170926,29601.12,8380,2435,17.32
9,Furnishings,385609,46967.43,11225,3170,12.18
2,Art,372163,57953.91,16301,4883,15.57
11,Machines,779071,58867.87,4906,1486,7.56
12,Paper,244307,59207.68,12822,3538,24.23
3,Binders,461952,72449.85,21429,6152,15.68


### Initial Finding

Product profitability varies substantially across MegaStore's portfolio.

Tables are the only sub-category generating an overall loss, producing approximately £757,000 in sales while losing more than £64,000, equivalent to a -8.47% profit margin. This indicates that relatively strong sales activity is not translating into profitable performance for this product group.

Machines also generate substantial revenue but operate at a comparatively weak 7.56% margin, while higher-margin sub-categories such as Paper and Copiers demonstrate that revenue alone does not provide a complete measure of commercial performance.

These results suggest that product portfolio decisions should consider both sales volume and margin rather than prioritising revenue alone.

## 9. Why are Tables losing money?

Investigate the cause rather than stopping at the symptom.
    
First suspect is discounting.

In [17]:
tables_analysis = (
    df[df["sub_category"] == "Tables"]
    .groupby("discount")
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        quantity=("quantity", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

tables_analysis["profit_margin_pct"] = (
    tables_analysis["profit"]
    / tables_analysis["sales"]
    * 100
)

display(
    tables_analysis.round({
        "discount": 2,
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2
    })
)

,discount,sales,profit,quantity,transactions,profit_margin_pct
0,0.00,261807,60447.20,716,208,23.09
1,0.10,3481,795.03,8,3,22.84
2,0.20,157721,7215.54,688,183,4.57
3,0.30,104747,-11960.48,413,116,-11.42
4,0.35,54159,-9122.65,169,45,-16.84
5,0.40,52162,-17105.18,313,85,-32.79
6,0.45,8032,-3596.30,56,13,-44.77
7,0.47,32502,-10680.28,125,34,-32.86
8,0.50,26505,-15049.86,187,52,-56.78
9,0.55,6308,-3150.67,36,10,-49.95


### Tables Discount Finding

Discounting is strongly associated with deteriorating profitability within the Tables sub-category.

Without a discount, Tables generated approximately £261,807 in sales and £60,447 in profit, representing a 23.09% profit margin. At a 20% discount, the sub-category remained profitable, but margin declined sharply to 4.57%.

From the observed 30% discount level onward, every discount group generated an overall loss, with profitability deteriorating further at many of the higher discount levels.

This suggests that aggressive discounting may be a major contributor to the poor overall performance of Tables. However, discounting should not be treated as the sole causal explanation without considering other factors such as product mix, geography and fulfilment costs.

## 10. Is this a Tables problem or a company-wide discount problem?
This is the important next question.
Analysing Tables only, conclusion could be too narrow. Test whether the same relationship appears across MegaStore as a whole.

Does MegaStore's discount strategy systematically erode profitability, and is there an observed discount range beyond which transactions become economically unattractive?

In [18]:
discount_profitability = (
    df.groupby("discount")
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        quantity=("quantity", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

discount_profitability["profit_margin_pct"] = (
    discount_profitability["profit"]
    / discount_profitability["sales"]
    * 100
)

display(
    discount_profitability.round({
        "discount": 2,
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2
    })
)

,discount,sales,profit,quantity,transactions,profit_margin_pct
0,0.00,6992734,1770695.27,98768,29009,25.32
1,0.00,261395,57976.58,1657,461,22.18
2,0.07,121620,21148.50,533,150,17.39
3,0.10,1579618,260641.71,15417,4068,16.50
4,0.15,306037,27375.90,2021,541,8.95
5,0.17,247404,28163.07,2720,735,11.38
6,0.20,1203855,117715.87,18653,4998,9.78
7,0.20,16213,-595.27,135,41,-3.67
8,0.25,88063,800.59,826,198,0.91
9,0.27,89351,-1675.08,1453,388,-1.87


### Company-Wide Discount Finding

The relationship between discounting and profitability extends beyond the Tables sub-category and appears across MegaStore's wider business.

Transactions with no discount generated approximately £6.99 million in sales and £1.77 million in profit, representing a 25.32% profit margin. Profit margins progressively weakened across many higher discount levels, with profitability approaching break-even around the mid-20% range.

From the observed discount groups displayed at approximately 27% and above, every group generated an overall loss, with increasingly severe negative margins at many of the highest discount levels.

This provides evidence of a strong association between aggressive discounting and profit erosion across MegaStore. However, the analysis does not establish discounting as the sole causal factor, and the appropriate discount threshold may differ across products and markets.

## 11. Financial Impact of High Discounting

The discount analysis indicates substantial deterioration in profitability at higher discount levels.

To quantify the commercial impact, transactions receiving discounts of 30% or more are examined as a high-discount group. This threshold is used as an analytical benchmark rather than a recommended pricing policy.

In [19]:
high_discount = df[df["discount"] >= 0.30]

high_discount_sales = high_discount["sales"].sum()
high_discount_profit = high_discount["profit"].sum()
high_discount_transactions = len(high_discount)

high_discount_margin = (
    high_discount_profit / high_discount_sales * 100
)

high_discount_sales_share = (
    high_discount_sales / df["sales"].sum() * 100
)

high_discount_transaction_share = (
    high_discount_transactions / len(df) * 100
)

print(f"High-discount transaction lines: {high_discount_transactions:,}")
print(f"Share of all transaction lines: {high_discount_transaction_share:.2f}%")

print(f"\nSales from high-discount transactions: £{high_discount_sales:,.2f}")
print(f"Share of total sales: {high_discount_sales_share:.2f}%")

print(f"\nProfit from high-discount transactions: £{high_discount_profit:,.2f}")
print(f"Profit margin: {high_discount_margin:.2f}%")

High-discount transaction lines: 10,701
Share of all transaction lines: 20.86%

Sales from high-discount transactions: £1,736,615.00
Share of total sales: 13.74%

Profit from high-discount transactions: £-813,212.32
Profit margin: -46.83%


### High-Discount Financial Impact

High discounting represents a significant area of commercial risk for MegaStore.

Transactions receiving discounts of 30% or more accounted for 10,701 transaction lines, equivalent to 20.86% of all transaction lines. Although these transactions generated approximately £1.74 million in sales, or 13.74% of total company revenue, they collectively produced a loss of approximately £813,212 and an overall profit margin of -46.83%.

This indicates that a meaningful portion of MegaStore's revenue is being generated under commercially weak conditions. High discounting therefore warrants closer management attention, although the results should not be interpreted as proof that discounting alone caused the losses.

## 12. Is discounting always bad?
Before recommending that management simply reduce discounts, carry out a more sophisticated test.
Some discounted transactions may still be profitable. Compare profitable versus loss-making transaction lines within the ≥30% discount group.

In [20]:
high_discount_breakdown = (
    high_discount
    .assign(
        profitability=np.where(
            high_discount["profit"] < 0,
            "Loss-making",
            "Profitable / Break-even"
        )
    )
    .groupby("profitability")
    .agg(
        transactions=("order_id", "size"),
        sales=("sales", "sum"),
        profit=("profit", "sum")
    )
    .reset_index()
)

high_discount_breakdown["transaction_share_pct"] = (
    high_discount_breakdown["transactions"]
    / len(high_discount)
    * 100
)

high_discount_breakdown["profit_margin_pct"] = (
    high_discount_breakdown["profit"]
    / high_discount_breakdown["sales"]
    * 100
)

display(
    high_discount_breakdown.round({
        "sales": 2,
        "profit": 2,
        "transaction_share_pct": 2,
        "profit_margin_pct": 2
    })
)

,profitability,transactions,sales,profit,transaction_share_pct,profit_margin_pct
0,Loss-making,9860,1543299,-831864.23,92.14,-53.90
1,Profitable / Break-even,841,193316,18651.91,7.86,9.65


### High-Discount Profitability Breakdown

High discounts are strongly associated with loss-making transactions, although they do not make losses inevitable.

Among the 10,701 transaction lines receiving discounts of 30% or more, 9,860 (92.14%) were loss-making. These transactions generated approximately £1.54 million in sales but produced approximately £831,864 in losses, representing a -53.90% profit margin.

Only 841 high-discount transaction lines (7.86%) were profitable or break-even. These generated approximately £193,316 in sales and £18,652 in profit.

Therefore, while some transactions remain profitable despite substantial discounts, the overwhelming majority of transactions within the 30%+ discount group are commercially unprofitable. This supports closer investigation of where high discounting is concentrated before recommending changes to discount policy.

## 13. Where is high discounting concentrated?
Connect discounting with geography.

Which markets are driving MegaStore's high-discount profit leakage?

In [21]:
high_discount_by_market = (
    high_discount
    .groupby("market")
    .agg(
        transactions=("order_id", "size"),
        sales=("sales", "sum"),
        profit=("profit", "sum")
    )
    .reset_index()
)

high_discount_by_market["profit_margin_pct"] = (
    high_discount_by_market["profit"]
    / high_discount_by_market["sales"]
    * 100
)

high_discount_by_market["share_of_high_discount_losses_pct"] = (
    high_discount_by_market["profit"].where(
        high_discount_by_market["profit"] < 0, 0
    ).abs()
    / high_discount_by_market["profit"].where(
        high_discount_by_market["profit"] < 0, 0
    ).abs().sum()
    * 100
)

high_discount_by_market = (
    high_discount_by_market
    .sort_values("profit")
)

display(
    high_discount_by_market.round({
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2,
        "share_of_high_discount_losses_pct": 2
    })
)

,market,transactions,sales,profit,profit_margin_pct,share_of_high_discount_losses_pct
3,EU,1597,354289,-164727.93,-46.50,20.26
0,APAC,2465,496532,-156273.98,-31.47,19.22
4,LATAM,2615,335415,-145284.50,-43.31,17.87
5,US,1393,362762,-135376.06,-37.32,16.65
2,EMEA,1609,127878,-122944.27,-96.14,15.12
1,Africa,1022,59739,-88605.59,-148.32,10.90


### Geographic Concentration of High-Discount Losses

High-discount profit leakage is present across every MegaStore market analysed, indicating that the issue is not isolated to a single geography.

The EU generated the largest absolute loss from transactions discounted by 30% or more, losing approximately £164,728, followed by APAC (£156,274), LATAM (£145,285), and the US (£135,376).

However, absolute losses do not tell the full story. Africa and EMEA displayed the weakest economics relative to their high-discount sales. Africa generated approximately £59,739 in high-discount sales while losing approximately £88,606, producing a -148.32% margin. EMEA generated approximately £127,878 in sales while losing approximately £122,944, equivalent to a -96.14% margin.

This suggests that management should distinguish between markets creating the largest absolute financial losses and markets where high-discount transactions exhibit the most severe margin deterioration.

## 14. Important control test
Be careful analytically.
Africa having a −148% margin on high-discount transactions does not mean Africa is an unprofitable market overall.
Comparing each market's overall performance against its high-discount performance.

In [22]:
market_performance = (
    df.groupby("market")
    .agg(
        total_sales=("sales", "sum"),
        total_profit=("profit", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

market_performance["overall_margin_pct"] = (
    market_performance["total_profit"]
    / market_performance["total_sales"]
    * 100
)

market_comparison = market_performance.merge(
    high_discount_by_market[
        ["market", "sales", "profit", "profit_margin_pct"]
    ],
    on="market",
    how="left"
)

market_comparison = market_comparison.rename(
    columns={
        "sales": "high_discount_sales",
        "profit": "high_discount_profit",
        "profit_margin_pct": "high_discount_margin_pct"
    }
)

market_comparison["high_discount_sales_share_pct"] = (
    market_comparison["high_discount_sales"]
    / market_comparison["total_sales"]
    * 100
)

market_comparison = market_comparison.sort_values(
    "overall_margin_pct",
    ascending=False
)

display(
    market_comparison.round({
        "total_sales": 2,
        "total_profit": 2,
        "overall_margin_pct": 2,
        "high_discount_sales": 2,
        "high_discount_profit": 2,
        "high_discount_margin_pct": 2,
        "high_discount_sales_share_pct": 2
    })
)

,market,total_sales,total_profit,transactions,overall_margin_pct,high_discount_sales,high_discount_profit,high_discount_margin_pct,high_discount_sales_share_pct
2,Canada,66932,17817.39,384,26.62,NaN,NaN,NaN,NaN
4,EU,2938139,372829.74,10000,12.69,354289.0,-164727.93,-46.50,12.06
6,US,2297354,286397.02,9994,12.47,362762.0,-135376.06,-37.32,15.79
0,APAC,3585833,437577.58,11002,12.20,496532.0,-156273.98,-31.47,13.85
1,Africa,783776,88871.63,4587,11.34,59739.0,-88605.59,-148.32,7.62
5,LATAM,2164687,221643.49,10294,10.24,335415.0,-145284.50,-43.31,15.49
3,EMEA,806184,43897.97,5029,5.45,127878.0,-122944.27,-96.14,15.86


### Overall Market vs High-Discount Performance

The comparison between overall market performance and high-discount activity shows that poor high-discount economics do not necessarily imply that the wider market is unprofitable.

Africa generated an overall profit margin of 11.34%, despite its transactions discounted by 30% or more producing a -148.32% margin. High-discount sales represented only 7.62% of Africa's total sales, suggesting that the severe losses are concentrated within a relatively small portion of the market rather than reflecting the profitability of Africa as a whole.

The EU, US, and APAC also remained profitable overall, with margins between approximately 12% and 13%, despite substantial losses within their high-discount transactions.

EMEA presents a broader concern. It recorded the lowest overall market margin at 5.45%, while its high-discount transactions produced a -96.14% margin and represented 15.86% of market sales.

Canada recorded the highest overall margin at 26.62% and had no transactions within the 30%+ high-discount group.

These results indicate that discount-related profit leakage should be investigated within each market rather than treating weak high-discount performance as evidence that an entire geographic market is commercially unattractive.

## 15. Isolate the effect more clearly

What would each market's performance look like when ≥30%-discount transactions are separated from the rest of its business?
How profitable is the rest of the existing business compared with the high-discount portion?

In [23]:
df["discount_group"] = np.where(
    df["discount"] >= 0.30,
    "30%+ Discount",
    "Below 30% Discount"
)

market_discount_comparison = (
    df.groupby(["market", "discount_group"])
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

market_discount_comparison["profit_margin_pct"] = (
    market_discount_comparison["profit"]
    / market_discount_comparison["sales"]
    * 100
)

market_discount_comparison = (
    market_discount_comparison
    .sort_values(["market", "discount_group"])
)

display(
    market_discount_comparison.round({
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2
    })
)

,market,discount_group,sales,profit,transactions,profit_margin_pct
0,APAC,30%+ Discount,496532,-156273.98,2465,-31.47
1,APAC,Below 30% Discount,3089301,593851.56,8537,19.22
2,Africa,30%+ Discount,59739,-88605.59,1022,-148.32
3,Africa,Below 30% Discount,724037,177477.22,3565,24.51
4,Canada,Below 30% Discount,66932,17817.39,384,26.62
5,EMEA,30%+ Discount,127878,-122944.27,1609,-96.14
6,EMEA,Below 30% Discount,678306,166842.24,3420,24.60
7,EU,30%+ Discount,354289,-164727.93,1597,-46.50
8,EU,Below 30% Discount,2583850,537557.67,8403,20.80
9,LATAM,30%+ Discount,335415,-145284.50,2615,-43.31


### Market Profitability by Discount Group

Separating transactions into below-30% and 30%+ discount groups reveals a consistent pattern across MegaStore's geographic markets.

Every market remained profitable within the below-30% discount group, with profit margins ranging from approximately 19% to 27%. In contrast, every market containing transactions discounted by 30% or more generated an overall loss within that high-discount group.

The deterioration was particularly severe in Africa, where the margin fell from 24.51% for transactions below 30% discount to -148.32% for transactions discounted by 30% or more. EMEA showed a similar pattern, declining from 24.60% to -96.14%.

APAC, EU, LATAM, and the US also showed substantial differences between their lower-discount and high-discount business.

These results provide strong evidence that high-discount activity is associated with profit leakage across multiple geographic markets rather than being confined to one region. However, the comparison remains observational and does not establish that discounting alone causes the difference in profitability.

## 16. Shipping Economics

Discounting explains an important component of MegaStore's profit leakage, but profitability may also be affected by fulfilment costs.

This section evaluates shipping economics to determine whether particular shipping modes create disproportionately high logistics costs or weaker commercial margins.

The analysis begins by comparing sales, profit, shipping costs, and profitability across shipping modes.

In [24]:
shipping_analysis = (
    df.groupby("ship_mode")
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        shipping_cost=("shipping_cost", "sum"),
        quantity=("quantity", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

shipping_analysis["profit_margin_pct"] = (
    shipping_analysis["profit"]
    / shipping_analysis["sales"]
    * 100
)

shipping_analysis["shipping_cost_pct_sales"] = (
    shipping_analysis["shipping_cost"]
    / shipping_analysis["sales"]
    * 100
)

shipping_analysis["avg_shipping_cost_per_transaction"] = (
    shipping_analysis["shipping_cost"]
    / shipping_analysis["transactions"]
)

shipping_analysis = shipping_analysis.sort_values(
    "profit_margin_pct"
)

display(
    shipping_analysis.round({
        "sales": 2,
        "profit": 2,
        "shipping_cost": 2,
        "profit_margin_pct": 2,
        "shipping_cost_pct_sales": 2,
        "avg_shipping_cost_per_transaction": 2
    })
)

,ship_mode,sales,profit,shipping_cost,quantity,transactions,profit_margin_pct,shipping_cost_pct_sales,avg_shipping_cost_per_transaction
0,First Class,1831067,208104.68,308103.25,26039,7505,11.37,16.83,41.05
1,Same Day,667202,76173.07,115974.06,9230,2701,11.42,17.38,42.94
2,Second Class,2565747,294161.06,314112.62,35724,10309,11.46,12.24,30.47
3,Standard Class,7578889,890596.02,614630.76,107319,30775,11.75,8.11,19.97


### Shipping Mode Finding

Shipping costs vary substantially across fulfilment methods, although overall profitability remains relatively similar across shipping modes.

Standard Class recorded the lowest shipping-cost burden, with shipping costs equivalent to 8.11% of sales and an average shipping cost of approximately £19.97 per transaction line. In comparison, Same Day shipping costs represented 17.38% of sales and averaged approximately £42.94 per transaction line.

Despite these differences, profit margins remained relatively consistent across all four shipping modes, ranging from 11.37% for First Class to 11.75% for Standard Class.

Therefore, faster shipping methods are associated with substantially higher fulfilment costs, but shipping mode alone does not appear to explain major differences in overall profitability. More detailed analysis is required to identify whether expensive fulfilment becomes problematic under particular commercial conditions.

## 17. Test the interaction between shipping and discounting

What happens when expensive fulfilment is combined with aggressive discounting?

In [25]:
shipping_discount_analysis = (
    df.groupby(["ship_mode", "discount_group"])
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        shipping_cost=("shipping_cost", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

shipping_discount_analysis["profit_margin_pct"] = (
    shipping_discount_analysis["profit"]
    / shipping_discount_analysis["sales"]
    * 100
)

shipping_discount_analysis["shipping_cost_pct_sales"] = (
    shipping_discount_analysis["shipping_cost"]
    / shipping_discount_analysis["sales"]
    * 100
)

display(
    shipping_discount_analysis.round({
        "sales": 2,
        "profit": 2,
        "shipping_cost": 2,
        "profit_margin_pct": 2,
        "shipping_cost_pct_sales": 2
    })
)

,ship_mode,discount_group,sales,profit,shipping_cost,transactions,profit_margin_pct,shipping_cost_pct_sales
0,First Class,30%+ Discount,258478,-131834.92,43857.32,1618,-51.00,16.97
1,First Class,Below 30% Discount,1572589,339939.60,264245.93,5887,21.62,16.80
2,Same Day,30%+ Discount,102011,-50157.41,16602.38,555,-49.17,16.28
3,Same Day,Below 30% Discount,565191,126330.48,99371.68,2146,22.35,17.58
4,Second Class,30%+ Discount,356799,-164229.48,42716.22,2170,-46.03,11.97
5,Second Class,Below 30% Discount,2208948,458390.54,271396.40,8139,20.75,12.29
6,Standard Class,30%+ Discount,1019327,-466990.51,80830.30,6358,-45.81,7.93
7,Standard Class,Below 30% Discount,6559562,1357586.54,533800.46,24417,20.70,8.14


### Shipping and Discount Interaction

The interaction between shipping mode and discounting shows that shipping method alone does not explain MegaStore's major profitability problems.

For transactions discounted below 30%, every shipping mode remained strongly profitable, with margins ranging from 20.70% to 22.35%. This was true even for faster and more expensive fulfilment methods such as First Class and Same Day.

In contrast, every shipping mode became heavily loss-making within the 30%+ discount group. Profit margins ranged from -45.81% for Standard Class to -51.00% for First Class.

Standard Class is particularly informative because its shipping costs represented only 7.93% of high-discount sales, substantially below the faster shipping methods, yet the group still generated a -45.81% profit margin.

These results suggest that higher fulfilment costs contribute to commercial cost pressure, but shipping mode alone is unlikely to explain the severe losses observed in heavily discounted transactions. The combination of commercial factors, particularly aggressive discounting, requires greater management attention.

## 18. Product Portfolio Strategy

The business question becomes:

Which product groups should management protect, improve, or urgently review?

In [26]:
median_subcategory_sales = subcategory_profitability["sales"].median()

overall_profit_margin = (
    df["profit"].sum()
    / df["sales"].sum()
    * 100
)

print(f"Median Sub-Category Sales: £{median_subcategory_sales:,.2f}")
print(f"Company Profit Margin Benchmark: {overall_profit_margin:.2f}%")

Median Sub-Category Sales: £749,307.00
Company Profit Margin Benchmark: 11.62%


## 19. Product Portfolio Strategy

Sales performance alone does not determine whether a product group is commercially attractive.

To evaluate MegaStore's product portfolio, sub-categories are classified using two benchmarks:

- Median sub-category sales of approximately £749,307
- Company-wide profit margin of 11.62%

This creates four portfolio groups that distinguish high- and lower-sales product areas according to whether their profit margins are above or below the company benchmark.

### The Product Portfolio Matrix
Classify each sub-category into four strategic groups:

| Portfolio Group | Meaning |
|---|---|
| **High Sales / Strong Margin** | Major revenue generators with above-company profitability |
| **High Sales / Weak Margin** | Strong revenue but below-company profitability |
| **Lower Sales / Strong Margin** | Smaller revenue contribution but attractive profitability |
| **Lower Sales / Weak Margin** | Lower revenue and below-company profitability |

In [27]:
portfolio = subcategory_profitability.copy()

portfolio["portfolio_group"] = np.select(
    [
        (portfolio["sales"] >= median_subcategory_sales) &
        (portfolio["profit_margin_pct"] >= overall_profit_margin),

        (portfolio["sales"] >= median_subcategory_sales) &
        (portfolio["profit_margin_pct"] < overall_profit_margin),

        (portfolio["sales"] < median_subcategory_sales) &
        (portfolio["profit_margin_pct"] >= overall_profit_margin),

        (portfolio["sales"] < median_subcategory_sales) &
        (portfolio["profit_margin_pct"] < overall_profit_margin)
    ],
    [
        "High Sales / Strong Margin",
        "High Sales / Weak Margin",
        "Lower Sales / Strong Margin",
        "Lower Sales / Weak Margin"
    ],
    default="Unclassified"
)

portfolio = portfolio.sort_values(
    ["portfolio_group", "sales"],
    ascending=[True, False]
)

display(
    portfolio[
        [
            "sub_category",
            "sales",
            "profit",
            "profit_margin_pct",
            "portfolio_group"
        ]
    ].round({
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2
    })
)

,sub_category,sales,profit,profit_margin_pct,portfolio_group
13,Phones,1706874,216717.01,12.70,High Sales / Strong Margin
6,Copiers,1509439,258567.55,17.13,High Sales / Strong Margin
1,Appliances,1011081,141680.59,14.01,High Sales / Strong Margin
0,Accessories,749307,129626.31,17.30,High Sales / Strong Margin
5,Chairs,1501682,141973.80,9.45,High Sales / Weak Margin
4,Bookcases,1466559,161924.42,11.04,High Sales / Weak Margin
14,Storage,1127124,108461.49,9.62,High Sales / Weak Margin
11,Machines,779071,58867.87,7.56,High Sales / Weak Margin
16,Tables,757034,-64083.39,-8.47,High Sales / Weak Margin
3,Binders,461952,72449.85,15.68,Lower Sales / Strong Margin


### Product Portfolio Finding

The portfolio matrix shows that revenue scale and profitability do not always move together across MegaStore's product portfolio.

Phones, Copiers, Appliances, and Accessories combine relatively high sales with profit margins above the company benchmark of 11.62%, making them commercially strong product groups. Copiers are particularly attractive, generating approximately £1.51 million in sales and £258,568 in profit at a 17.13% margin.

In contrast, Chairs, Bookcases, Storage, Machines, and Tables generate relatively high sales but operate below the company-wide margin benchmark. Tables represent the most significant concern because they generated approximately £757,000 in sales while producing a -8.47% margin.

Several lower-sales sub-categories demonstrate attractive profitability. Paper generated a 24.23% margin, while Labels, Envelopes, Binders, and Art also achieved margins above the company benchmark.

These results suggest that portfolio management should consider both revenue scale and profit efficiency. High-revenue, weak-margin categories warrant margin-improvement initiatives, while strong-margin product groups may offer opportunities for commercially sustainable growth.

## 20. Move from sub-categories to individual products

It will be a huge mistake saying “Tables are bad” is too broad. Some individual Tables products could be highly profitable while a smaller group is responsible for most of the losses.

So next business question is:

Are MegaStore's product losses broadly distributed, or concentrated among a relatively small number of products?

In [28]:
product_profitability = (
    df.groupby(["product_id", "product_name", "sub_category"])
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        quantity=("quantity", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

product_profitability["profit_margin_pct"] = np.where(
    product_profitability["sales"] != 0,
    product_profitability["profit"]
    / product_profitability["sales"] * 100,
    np.nan
)

loss_products = product_profitability[
    product_profitability["profit"] < 0
].copy()

print(f"Unique products analysed: {len(product_profitability):,}")
print(f"Loss-making products: {len(loss_products):,}")
print(
    f"Loss-making product share: "
    f"{len(loss_products) / len(product_profitability) * 100:.2f}%"
)

print(
    f"\nSales from loss-making products: "
    f"£{loss_products['sales'].sum():,.2f}"
)

print(
    f"Combined profit from loss-making products: "
    f"£{loss_products['profit'].sum():,.2f}"
)

print("\n10 products with the largest total losses:")

display(
    loss_products
    .sort_values("profit")
    .head(10)
    [
        [
            "product_name",
            "sub_category",
            "sales",
            "profit",
            "profit_margin_pct",
            "transactions"
        ]
    ]
    .round({
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2
    })
)

Unique products analysed: 10,768
Loss-making products: 3,032
Loss-making product share: 28.16%

Sales from loss-making products: £3,247,801.00
Combined profit from loss-making products: £-557,776.73

10 products with the largest total losses:


,product_name,sub_category,sales,profit,profit_margin_pct,transactions
9574,Cubify CubeX 3D Printer Double Head Print,Machines,11100,-8879.97,-80.00,3
2614,"Hoover Stove, White",Appliances,11730,-4958.16,-42.27,6
9605,Lexmark MX611dhe Monochrome Laser Printer,Machines,16830,-4589.97,-27.27,4
10424,"Apple Smart Phone, Full Size",Phones,7258,-4574.64,-63.03,4
9984,"Motorola Smart Phone, Cordless",Phones,10350,-3998.68,-38.63,7
9876,Cubify CubeX 3D Printer Triple Head Print,Machines,8000,-3839.99,-48.00,1
699,"Office Star Executive Leather Armchair, Black",Chairs,6498,-3066.78,-47.20,5
1993,Chromcraft Bull-Nose Wood Oval Conference Tabl...,Tables,9918,-2876.12,-29.00,5
2157,"Lesro Computer Table, Fully Assembled",Tables,1200,-2798.49,-233.21,3
2118,"Hon Conference Table, Rectangular",Tables,3667,-2619.31,-71.43,3


### Individual Product Profitability Finding

Product-level analysis reveals that profit leakage extends beyond the Tables sub-category and is concentrated across specific products within several parts of the portfolio.

Of 10,768 unique product records analysed, 3,032 (28.16%) were loss-making overall. These products generated approximately £3.25 million in sales but collectively produced approximately £557,777 in losses.

The largest individual loss was generated by the Cubify CubeX 3D Printer Double Head Print, which produced approximately £11,100 in sales but lost £8,880, equivalent to an -80.00% margin.

Importantly, the largest product losses were distributed across Machines, Appliances, Phones, Chairs, and Tables. This demonstrates that profitable sub-categories can still contain individual products with poor commercial performance.

Product portfolio decisions should therefore combine sub-category-level performance with product-level diagnostics rather than assuming that every product within a profitable sub-category contributes positively to overall profitability.

## 21. Are product losses concentrated?

Is most product-level profit leakage coming from a relatively small number of products?

This matters because if a small number of products account for a large share of the £557,777 loss, management can prioritise those products instead of reviewing all 3,032 loss-making products equally.

In [29]:
loss_concentration = (
    loss_products
    .sort_values("profit")
    .copy()
)

# Convert negative profit into a positive loss amount
loss_concentration["loss_amount"] = (
    loss_concentration["profit"].abs()
)

total_product_losses = loss_concentration["loss_amount"].sum()

loss_concentration["cumulative_loss"] = (
    loss_concentration["loss_amount"].cumsum()
)

loss_concentration["cumulative_loss_pct"] = (
    loss_concentration["cumulative_loss"]
    / total_product_losses
    * 100
)

for n in [10, 50, 100, 500]:
    subset_loss = loss_concentration.head(n)["loss_amount"].sum()
    share = subset_loss / total_product_losses * 100

    print(
        f"Top {n} loss-making products: "
        f"£{subset_loss:,.2f} "
        f"({share:.2f}% of product-level losses)"
    )

products_for_50pct = (
    loss_concentration["cumulative_loss_pct"] < 50
).sum() + 1

products_for_80pct = (
    loss_concentration["cumulative_loss_pct"] < 80
).sum() + 1

print(
    f"\nProducts accounting for 50% of losses: "
    f"{products_for_50pct:,}"
)

print(
    f"Products accounting for 80% of losses: "
    f"{products_for_80pct:,}"
)

Top 10 loss-making products: £42,202.12 (7.57% of product-level losses)
Top 50 loss-making products: £113,633.99 (20.37% of product-level losses)
Top 100 loss-making products: £172,321.13 (30.89% of product-level losses)
Top 500 loss-making products: £384,504.17 (68.94% of product-level losses)

Products accounting for 50% of losses: 246
Products accounting for 80% of losses: 770


### Product Loss Concentration Finding

Product-level losses are concentrated within a smaller subset of the loss-making portfolio, although they are not dominated by only a handful of products.

The 10 products with the largest losses accounted for 7.57% of total product-level losses, while the top 100 accounted for 30.89%. The top 500 accounted for 68.94%.

More importantly, only 246 products accounted for 50% of total product-level losses. This represents approximately 8.1% of the 3,032 loss-making products. Similarly, 770 products, approximately 25.4% of the loss-making product set, accounted for 80% of losses.

This concentration provides management with a practical prioritisation strategy. Rather than reviewing all loss-making products equally, commercial investigation could initially focus on the relatively small group responsible for the majority of product-level losses.

These products should be examined for factors such as pricing, discount levels, product economics, geographic performance, and fulfilment costs before decisions are made to reprice, restrict discounting, renegotiate costs, or rationalise parts of the portfolio.

## 22. Growth Quality

Is MegaStore's sales growth translating into healthy profit growth?

Compare **revenue growth against profit growth and margins.**

In [30]:
yearly_performance = (
    df.groupby("year")
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        shipping_cost=("shipping_cost", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

yearly_performance["profit_margin_pct"] = (
    yearly_performance["profit"]
    / yearly_performance["sales"]
    * 100
)

yearly_performance["sales_growth_pct"] = (
    yearly_performance["sales"]
    .pct_change()
    * 100
)

yearly_performance["profit_growth_pct"] = (
    yearly_performance["profit"]
    .pct_change()
    * 100
)

display(
    yearly_performance.round({
        "sales": 2,
        "profit": 2,
        "shipping_cost": 2,
        "profit_margin_pct": 2,
        "sales_growth_pct": 2,
        "profit_growth_pct": 2
    })
)

,year,sales,profit,shipping_cost,transactions,profit_margin_pct,sales_growth_pct,profit_growth_pct
0,2011,2259511,248940.81,244271.08,8998,11.02,NaN,NaN
1,2012,2677493,307415.28,283491.89,10962,11.48,18.50,23.49
2,2013,3405860,408512.76,364549.95,13799,11.99,27.20,32.89
3,2014,4300041,504165.97,460507.77,17531,11.72,26.25,23.41


### Growth Quality Finding

MegaStore experienced substantial commercial growth between 2011 and 2014, with annual sales increasing from approximately £2.26 million to £4.30 million and annual profit increasing from approximately £248,941 to £504,166.

Growth quality was particularly strong in 2012 and 2013. In 2012, sales increased by 18.50% while profit grew by 23.49%. In 2013, sales increased by 27.20% while profit grew by 32.89%. As a result, the company's profit margin improved from 11.02% in 2011 to 11.99% in 2013.

However, the pattern changed slightly in 2014. Sales grew by 26.25%, while profit increased by a slower 23.41%. Consequently, profit margin declined from 11.99% to 11.72%.

Although 2014 remained strongly profitable and delivered substantial growth, the slight margin compression indicates that revenue growth did not translate into profit at the same rate as in the previous two years.

Overall, MegaStore demonstrated profitable growth across the period, but the 2014 result highlights the importance of monitoring growth quality rather than evaluating commercial performance through revenue growth alone.

## 23. Investigate the 2014 margin compression

Test whether the proportion of business conducted at high discounts changed over time.

In [31]:
yearly_discount_mix = (
    df.groupby(["year", "discount_group"])
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

year_totals = (
    df.groupby("year")
    .agg(
        total_sales=("sales", "sum"),
        total_transactions=("order_id", "size")
    )
    .reset_index()
)

yearly_discount_mix = yearly_discount_mix.merge(
    year_totals,
    on="year"
)

yearly_discount_mix["sales_share_pct"] = (
    yearly_discount_mix["sales"]
    / yearly_discount_mix["total_sales"]
    * 100
)

yearly_discount_mix["transaction_share_pct"] = (
    yearly_discount_mix["transactions"]
    / yearly_discount_mix["total_transactions"]
    * 100
)

yearly_discount_mix["profit_margin_pct"] = (
    yearly_discount_mix["profit"]
    / yearly_discount_mix["sales"]
    * 100
)

display(
    yearly_discount_mix[
        [
            "year",
            "discount_group",
            "sales",
            "profit",
            "transactions",
            "sales_share_pct",
            "transaction_share_pct",
            "profit_margin_pct"
        ]
    ].round(2)
)

,year,discount_group,sales,profit,transactions,sales_share_pct,transaction_share_pct,profit_margin_pct
0,2011,30%+ Discount,350902,-150248.58,1901,15.53,21.13,-42.82
1,2011,Below 30% Discount,1908609,399189.40,7097,84.47,78.87,20.92
2,2012,30%+ Discount,342528,-173624.38,2236,12.79,20.40,-50.69
3,2012,Below 30% Discount,2334965,481039.66,8726,87.21,79.60,20.60
4,2013,30%+ Discount,464034,-216526.23,2854,13.62,20.68,-46.66
5,2013,Below 30% Discount,2941826,625038.99,10945,86.38,79.32,21.25
6,2014,30%+ Discount,579151,-272813.13,3710,13.47,21.16,-47.11
7,2014,Below 30% Discount,3720890,776979.10,13821,86.53,78.84,20.88


### Yearly Discount Mix Finding

The slight decline in MegaStore's profit margin in 2014 does not appear to have been driven by a major increase in the proportion of sales generated through 30%+ discounts.

High-discount sales represented 15.53% of total sales in 2011, 12.79% in 2012, 13.62% in 2013, and 13.47% in 2014. The transaction share of the high-discount group was also relatively stable, ranging from approximately 20% to 21% throughout the period.

Nevertheless, heavily discounted transactions remained consistently loss-making. In 2014, the 30%+ discount group generated approximately £579,151 in sales but produced a loss of approximately £272,813, equivalent to a -47.11% profit margin. In comparison, transactions below the 30% analytical threshold generated approximately £3.72 million in sales and £776,979 in profit at a 20.88% margin.

Therefore, aggressive discounting remained an important source of profit leakage, but changes in the high-discount sales mix do not by themselves explain the slight company-wide margin compression observed in 2014. Other commercial or cost factors should be investigated.

## 24. Investigate shipping-cost pressure over time

In [32]:
yearly_shipping = (
    df.groupby("year")
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        shipping_cost=("shipping_cost", "sum"),
        transactions=("order_id", "size"),
        quantity=("quantity", "sum")
    )
    .reset_index()
)

yearly_shipping["shipping_cost_pct_sales"] = (
    yearly_shipping["shipping_cost"]
    / yearly_shipping["sales"]
    * 100
)

yearly_shipping["avg_shipping_cost_per_transaction"] = (
    yearly_shipping["shipping_cost"]
    / yearly_shipping["transactions"]
)

yearly_shipping["shipping_cost_per_unit"] = (
    yearly_shipping["shipping_cost"]
    / yearly_shipping["quantity"]
)

yearly_shipping["shipping_cost_growth_pct"] = (
    yearly_shipping["shipping_cost"]
    .pct_change()
    * 100
)

display(
    yearly_shipping.round({
        "sales": 2,
        "profit": 2,
        "shipping_cost": 2,
        "shipping_cost_pct_sales": 2,
        "avg_shipping_cost_per_transaction": 2,
        "shipping_cost_per_unit": 2,
        "shipping_cost_growth_pct": 2
    })
)

,year,sales,profit,shipping_cost,transactions,quantity,shipping_cost_pct_sales,avg_shipping_cost_per_transaction,shipping_cost_per_unit,shipping_cost_growth_pct
0,2011,2259511,248940.81,244271.08,8998,31443,10.81,27.15,7.77,NaN
1,2012,2677493,307415.28,283491.89,10962,38111,10.59,25.86,7.44,16.06
2,2013,3405860,408512.76,364549.95,13799,48136,10.70,26.42,7.57,28.59
3,2014,4300041,504165.97,460507.77,17531,60622,10.71,26.27,7.60,26.32


### Shipping Cost Trend Finding

Shipping costs increased substantially in absolute terms as MegaStore expanded, rising from approximately £244,271 in 2011 to £460,508 in 2014. However, the relative cost of fulfilment remained remarkably stable.

Shipping costs represented 10.81% of sales in 2011, 10.59% in 2012, 10.70% in 2013, and 10.71% in 2014. Average shipping cost per transaction line also remained relatively stable and declined slightly from £26.42 in 2013 to £26.27 in 2014.

Shipping cost per unit increased only marginally from £7.57 in 2013 to £7.60 in 2014.

Between 2013 and 2014, total shipping costs increased by approximately 26.32%, broadly in line with sales growth of 26.25%.

Therefore, there is little evidence that increasing fulfilment-cost intensity was responsible for the slight decline in company-wide profit margin in 2014. The margin movement is more likely to reflect changes in the mix of products, markets, orders, or other commercial factors rather than a substantial deterioration in shipping efficiency.

## 25. Product Mix and the 2014 Margin Compression

Discount mix: broadly stable.

Shipping-cost intensity: broadly stable.

investigate product mix.

The important question isn't simply which sub-category grew most: Did 2014 growth shift toward lower-margin or loss-making sub-categories?

In [33]:
subcategory_year = (
    df.groupby(["year", "sub_category"])
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum")
    )
    .reset_index()
)

subcategory_year["profit_margin_pct"] = np.where(
    subcategory_year["sales"] != 0,
    subcategory_year["profit"]
    / subcategory_year["sales"] * 100,
    np.nan
)

# Compare 2013 with 2014
product_mix_2013_2014 = (
    subcategory_year[
        subcategory_year["year"].isin([2013, 2014])
    ]
    .pivot(
        index="sub_category",
        columns="year",
        values=["sales", "profit", "profit_margin_pct"]
    )
)

product_mix_2013_2014.columns = [
    f"{metric}_{year}"
    for metric, year in product_mix_2013_2014.columns
]

product_mix_2013_2014 = (
    product_mix_2013_2014
    .reset_index()
)

product_mix_2013_2014["sales_growth_pct"] = (
    (
        product_mix_2013_2014["sales_2014"]
        / product_mix_2013_2014["sales_2013"]
    ) - 1
) * 100

product_mix_2013_2014["profit_growth_pct"] = (
    (
        product_mix_2013_2014["profit_2014"]
        / product_mix_2013_2014["profit_2013"]
    ) - 1
) * 100

product_mix_2013_2014["sales_increase"] = (
    product_mix_2013_2014["sales_2014"]
    - product_mix_2013_2014["sales_2013"]
)

product_mix_2013_2014 = (
    product_mix_2013_2014
    .sort_values("sales_increase", ascending=False)
)

display(
    product_mix_2013_2014.round(2)
)

,sub_category,sales_2013,sales_2014,profit_2013,profit_2014,profit_margin_pct_2013,profit_margin_pct_2014,sales_growth_pct,profit_growth_pct,sales_increase
4,Bookcases,376037.0,513184.0,43049.24,63219.05,11.45,12.32,36.47,46.85,137147.0
6,Copiers,415510.0,550382.0,72300.69,104048.54,17.40,18.90,32.46,43.91,134872.0
1,Appliances,254955.0,359791.0,41485.52,53040.50,16.27,14.74,41.12,27.85,104836.0
13,Phones,453529.0,552013.0,46908.83,70657.64,10.34,12.80,21.72,50.63,98484.0
14,Storage,309492.0,383443.0,32011.03,39016.95,10.34,10.18,23.89,21.89,73951.0
5,Chairs,427522.0,493380.0,42027.02,41248.27,9.83,8.36,15.40,-1.85,65858.0
11,Machines,198373.0,260286.0,15611.66,18627.75,7.87,7.16,31.21,19.32,61913.0
0,Accessories,209916.0,253517.0,38805.42,41593.93,18.49,16.41,20.77,7.19,43601.0
16,Tables,202359.0,243460.0,-14040.49,-30545.91,-6.94,-12.55,20.31,117.56,41101.0
3,Binders,121086.0,160441.0,21523.63,22356.49,17.78,13.93,32.50,3.87,39355.0


### Product Mix and 2014 Margin Finding

The 2014 margin compression cannot be attributed to a simple shift toward unprofitable product categories. Several commercially strong sub-categories contributed substantially to sales growth.

Bookcases generated approximately £137,147 of additional sales while improving its profit margin from 11.45% to 12.32%. Copiers added approximately £134,872 of sales and improved its margin from 17.40% to 18.90%. Phones also delivered positive growth quality, adding approximately £98,484 of sales while increasing its margin from 10.34% to 12.80%.

However, some of MegaStore's additional 2014 revenue came from product groups with weaker economics.

Chairs added approximately £65,858 in sales, but profit declined by 1.85%, causing margin to fall from 9.83% to 8.36%. Machines added approximately £61,913 in sales while margin declined from 7.87% to 7.16%.

Tables presented the clearest deterioration. Sales increased by approximately £41,101, but the sub-category's margin declined from -6.94% to -12.55%, with annual losses increasing from approximately £14,040 to £30,546.

The evidence therefore suggests a mixed growth-quality effect in 2014. Strong growth in profitable product groups was accompanied by additional revenue from weaker-margin and increasingly loss-making areas. This product mix likely contributed to the slight company-wide margin compression, although the analysis does not establish product mix as its sole cause.

## Step 26. Geographic Growth Quality

Which markets are driving profitable growth, and which are expanding with weaker economics?

Compare 2013 versus 2014 by market:

In [34]:
market_year = (
    df.groupby(["year", "market"])
    .agg(
        sales=("sales", "sum"),
        profit=("profit", "sum"),
        transactions=("order_id", "size")
    )
    .reset_index()
)

market_year["profit_margin_pct"] = np.where(
    market_year["sales"] != 0,
    market_year["profit"]
    / market_year["sales"] * 100,
    np.nan
)

market_growth = (
    market_year[
        market_year["year"].isin([2013, 2014])
    ]
    .pivot(
        index="market",
        columns="year",
        values=["sales", "profit", "profit_margin_pct"]
    )
)

market_growth.columns = [
    f"{metric}_{year}"
    for metric, year in market_growth.columns
]

market_growth = market_growth.reset_index()

market_growth["sales_growth_pct"] = (
    market_growth["sales_2014"]
    / market_growth["sales_2013"] - 1
) * 100

market_growth["profit_growth_pct"] = (
    market_growth["profit_2014"]
    / market_growth["profit_2013"] - 1
) * 100

market_growth["sales_increase"] = (
    market_growth["sales_2014"]
    - market_growth["sales_2013"]
)

market_growth["profit_increase"] = (
    market_growth["profit_2014"]
    - market_growth["profit_2013"]
)

market_growth = market_growth.sort_values(
    "sales_increase",
    ascending=False
)

display(
    market_growth.round(2)
)

,market,sales_2013,sales_2014,profit_2013,profit_2014,profit_margin_pct_2013,profit_margin_pct_2014,sales_growth_pct,profit_growth_pct,sales_increase,profit_increase
4,EU,761686.0,1042224.0,98275.18,128944.36,12.90,12.37,36.83,31.21,280538.0,30669.17
0,APAC,974618.0,1209210.0,124680.66,140454.50,12.79,11.62,24.07,12.65,234592.0,15773.84
6,US,608523.0,734016.0,81726.93,93507.51,13.43,12.74,20.62,14.41,125493.0,11780.58
5,LATAM,608148.0,706691.0,61415.37,73334.80,10.10,10.38,16.20,19.41,98543.0,11919.43
3,EMEA,204654.0,301702.0,10598.04,22600.32,5.18,7.49,47.42,113.25,97048.0,12002.28
1,Africa,229069.0,283034.0,26687.11,39331.47,11.65,13.90,23.56,47.38,53965.0,12644.36
2,Canada,19162.0,23164.0,5129.46,5993.01,26.77,25.87,20.89,16.84,4002.0,863.55


### Geographic Growth Quality Finding

All MegaStore markets increased both sales and profit between 2013 and 2014, but the quality of growth varied substantially.

EU generated the largest absolute sales increase, adding approximately £280,538 in revenue and £30,669 in profit. However, sales grew by 36.83% compared with profit growth of 31.21%, causing profit margin to decline slightly from 12.90% to 12.37%.

APAC demonstrated a more pronounced divergence. Sales increased by approximately £234,592, representing growth of 24.07%, while profit increased by only 12.65%. Consequently, APAC's margin declined from 12.79% to 11.62%.

The US showed a similar pattern, with sales growth of 20.62% exceeding profit growth of 14.41%, reducing margin from 13.43% to 12.74%.

In contrast, several markets improved the quality of their growth. Africa increased sales by 23.56% while profit grew by 47.38%, raising its margin from 11.65% to 13.90%. LATAM also improved its margin from 10.10% to 10.38%.

EMEA recorded the fastest percentage sales growth at 47.42% and more than doubled profit, improving its margin from 5.18% to 7.49%. However, despite this improvement, EMEA remained a relatively low-margin market.

The geographic analysis therefore provides additional context for MegaStore's slight company-wide margin compression in 2014. Some of the largest absolute revenue growth occurred in APAC, EU, and the US, where profit grew more slowly than sales. Meanwhile, Africa and EMEA improved their profit efficiency considerably.

## 27. Order Economics

Are some high-value orders generating impressive revenue but poor or negative profit?

There is one important technical point: **order_id** repeats because one order can contain several transaction lines. Aggregate to the actual order level before analysing order economics.

In [35]:
order_economics = (
    df.groupby("order_id")
    .agg(
        order_sales=("sales", "sum"),
        order_profit=("profit", "sum"),
        quantity=("quantity", "sum"),
        shipping_cost=("shipping_cost", "sum"),
        avg_discount=("discount", "mean"),
        transaction_lines=("order_id", "size")
    )
    .reset_index()
)

order_economics["profit_margin_pct"] = np.where(
    order_economics["order_sales"] != 0,
    order_economics["order_profit"]
    / order_economics["order_sales"] * 100,
    np.nan
)

order_economics["shipping_cost_pct_sales"] = np.where(
    order_economics["order_sales"] != 0,
    order_economics["shipping_cost"]
    / order_economics["order_sales"] * 100,
    np.nan
)

loss_orders = order_economics[
    order_economics["order_profit"] < 0
].copy()

print(f"Total orders: {len(order_economics):,}")
print(f"Loss-making orders: {len(loss_orders):,}")
print(
    f"Loss-making order rate: "
    f"{len(loss_orders) / len(order_economics) * 100:.2f}%"
)

print(
    f"Sales from loss-making orders: "
    f"£{loss_orders['order_sales'].sum():,.2f}"
)

print(
    f"Combined profit from loss-making orders: "
    f"£{loss_orders['order_profit'].sum():,.2f}"
)

print("\n10 highest-sales loss-making orders:")

display(
    loss_orders
    .sort_values("order_sales", ascending=False)
    .head(10)
    .round({
        "order_sales": 2,
        "order_profit": 2,
        "shipping_cost": 2,
        "avg_discount": 2,
        "profit_margin_pct": 2,
        "shipping_cost_pct_sales": 2
    })
)

Total orders: 25,035
Loss-making orders: 6,124
Loss-making order rate: 24.46%
Sales from loss-making orders: £2,358,343.00
Combined profit from loss-making orders: £-843,609.63

10 highest-sales loss-making orders:


,order_id,order_sales,order_profit,quantity,shipping_cost,avg_discount,transaction_lines,profit_margin_pct,shipping_cost_pct_sales
888,CA-2011-145317,23660,-1789.73,25,87.26,0.29,7,-7.56,0.37
826,CA-2011-139892,10540,-1878.79,37,460.47,0.33,7,-17.83,4.37
24890,US-2014-168116,8167,-3825.34,6,707.29,0.35,2,-46.84,8.66
15421,IT-2013-4602742,6307,-433.79,22,226.60,0.05,4,-6.88,3.59
9991,ID-2013-70122,6119,-368.14,23,650.33,0.27,6,-6.02,10.63
22847,US-2011-106992,5580,-761.99,10,588.86,0.40,2,-13.66,10.55
10691,IN-2011-13030,5175,-145.19,21,318.26,0.10,5,-2.81,6.15
2242,CA-2013-108196,5017,-6892.37,10,513.05,0.53,3,-137.38,10.23
15485,IT-2014-1036058,4818,-714.72,29,217.41,0.26,8,-14.83,4.51
4450,CA-2014-158379,4750,-1030.64,10,410.19,0.37,3,-21.70,8.64


### Order Economics Finding

Order-level analysis shows that a substantial share of MegaStore's orders generate revenue without generating positive profit.

Of 25,035 orders analysed, 6,124 were loss-making, representing 24.46% of all orders. These orders generated approximately £2.36 million in sales but collectively produced approximately £843,610 in losses.

Several high-value orders demonstrate why revenue alone is an incomplete measure of commercial performance. For example, order CA-2011-145317 generated approximately £23,660 in sales but produced a loss of approximately £1,790.

More extreme cases also exist. Order CA-2013-108196 generated approximately £5,017 in sales but lost approximately £6,892, resulting in a -137.38% profit margin.

High discounting appears frequently among poorly performing orders, but it does not explain every case. Some loss-making orders had relatively modest average discounts, indicating that product mix, pricing, shipping costs, geography, and other order-level factors may also contribute to weak economics.

Management should therefore evaluate major orders using both revenue and profitability measures rather than treating high order value as evidence of commercial success.

## 28. Concentration of Order-Level Losse

Are the £843,610 of order-level losses spread broadly across 6,124 orders, or is a relatively small number of orders responsible for most of the damage?

In [36]:
order_loss_concentration = (
    loss_orders
    .sort_values("order_profit")
    .copy()
)

order_loss_concentration["loss_amount"] = (
    order_loss_concentration["order_profit"].abs()
)

total_order_losses = (
    order_loss_concentration["loss_amount"].sum()
)

order_loss_concentration["cumulative_loss"] = (
    order_loss_concentration["loss_amount"].cumsum()
)

order_loss_concentration["cumulative_loss_pct"] = (
    order_loss_concentration["cumulative_loss"]
    / total_order_losses
    * 100
)

for n in [10, 50, 100, 500, 1000]:
    subset_loss = (
        order_loss_concentration
        .head(n)["loss_amount"]
        .sum()
    )

    share = (
        subset_loss
        / total_order_losses
        * 100
    )

    print(
        f"Top {n} loss-making orders: "
        f"£{subset_loss:,.2f} "
        f"({share:.2f}% of order-level losses)"
    )

orders_for_50pct = (
    order_loss_concentration["cumulative_loss_pct"] < 50
).sum() + 1

orders_for_80pct = (
    order_loss_concentration["cumulative_loss_pct"] < 80
).sum() + 1

print(
    f"\nOrders accounting for 50% of losses: "
    f"{orders_for_50pct:,}"
)

print(
    f"Orders accounting for 80% of losses: "
    f"{orders_for_80pct:,}"
)

Top 10 loss-making orders: £36,871.82 (4.37% of order-level losses)
Top 50 loss-making orders: £106,989.73 (12.68% of order-level losses)
Top 100 loss-making orders: £164,690.72 (19.52% of order-level losses)
Top 500 loss-making orders: £411,011.19 (48.72% of order-level losses)
Top 1000 loss-making orders: £563,350.92 (66.78% of order-level losses)

Orders accounting for 50% of losses: 528
Orders accounting for 80% of losses: 1,626


### Order-Level Loss Concentration Finding

Order-level losses are meaningfully concentrated within a subset of MegaStore's loss-making orders.

The 10 largest loss-making orders accounted for only 4.37% of total order-level losses, while the top 100 accounted for 19.52%. However, the top 500 loss-making orders accounted for 48.72% of all order-level losses.

Only 528 orders were required to account for 50% of total order-level losses. This represents approximately 8.62% of the 6,124 loss-making orders.

Similarly, 1,626 orders accounted for 80% of order-level losses, representing approximately 26.55% of loss-making orders.

The losses are therefore not dominated by only a handful of extreme transactions, but they are sufficiently concentrated to support a targeted management approach.

Rather than reviewing every loss-making order equally, MegaStore could prioritise orders responsible for the largest cumulative losses and investigate their discount levels, product mix, shipping costs, geographic characteristics, and other commercial factors. This could support an exception-based order review process designed to identify high-risk transactions before similar losses recur.

## 29. What distinguishes loss-making orders?

What characteristics distinguish loss-making orders from profitable orders?

Compare discounting, shipping costs, order value, quantity and order complexity.

In [37]:
order_economics["order_status"] = np.where(
    order_economics["order_profit"] < 0,
    "Loss-making",
    "Profitable / Break-even"
)

order_status_comparison = (
    order_economics.groupby("order_status")
    .agg(
        orders=("order_id", "count"),
        total_sales=("order_sales", "sum"),
        total_profit=("order_profit", "sum"),
        avg_order_value=("order_sales", "mean"),
        avg_order_profit=("order_profit", "mean"),
        avg_discount=("avg_discount", "mean"),
        avg_shipping_cost=("shipping_cost", "mean"),
        avg_quantity=("quantity", "mean"),
        avg_transaction_lines=("transaction_lines", "mean")
    )
    .reset_index()
)

order_status_comparison["overall_profit_margin_pct"] = (
    order_status_comparison["total_profit"]
    / order_status_comparison["total_sales"]
    * 100
)

display(
    order_status_comparison.round({
        "total_sales": 2,
        "total_profit": 2,
        "avg_order_value": 2,
        "avg_order_profit": 2,
        "avg_discount": 3,
        "avg_shipping_cost": 2,
        "avg_quantity": 2,
        "avg_transaction_lines": 2,
        "overall_profit_margin_pct": 2
    })
)

,order_status,orders,total_sales,total_profit,avg_order_value,avg_order_profit,avg_discount,avg_shipping_cost,avg_quantity,avg_transaction_lines,overall_profit_margin_pct
0,Loss-making,6124,2358343,-843609.63,385.10,-137.75,0.437,40.99,7.19,2.10,-35.77
1,Profitable / Break-even,18911,10284562,2312644.45,543.84,122.29,0.046,58.26,7.10,2.03,22.49


### Loss-Making vs Profitable Order Characteristics

Comparison of loss-making and profitable orders reveals a substantial difference in discount behaviour.

Loss-making orders received an average discount of approximately 43.7%, compared with only 4.6% for profitable or break-even orders. This is the clearest distinguishing characteristic identified in the order-level comparison and is consistent with the broader discount profitability analysis.

Loss-making orders also had a lower average order value, approximately £385 compared with £544 for profitable or break-even orders.

Average shipping cost was lower among loss-making orders (£40.99) than among profitable or break-even orders (£58.26). Therefore, higher absolute shipping cost does not appear to be the primary characteristic distinguishing the two groups.

Order size and complexity were also very similar. Loss-making orders averaged 7.19 units and 2.10 transaction lines, compared with 7.10 units and 2.03 transaction lines for profitable or break-even orders.

Overall, loss-making orders generated a -35.77% profit margin, while profitable or break-even orders generated an aggregate margin of approximately 22.49%.

These results show a strong association between aggressive discounting and poor order economics. However, the analysis is descriptive and should not be interpreted as proving that discounting alone causes an order to become unprofitable.

## 30. Discount exposure within loss-making orders

How much of the loss-making order problem is associated with heavily discounted orders?

In [38]:
# Calculate sales-weighted discount at order level
df["discount_x_sales"] = df["discount"] * df["sales"]

weighted_order_discount = (
    df.groupby("order_id")
    .agg(
        total_sales=("sales", "sum"),
        weighted_discount_value=("discount_x_sales", "sum")
    )
    .reset_index()
)

weighted_order_discount["weighted_avg_discount"] = np.where(
    weighted_order_discount["total_sales"] != 0,
    weighted_order_discount["weighted_discount_value"]
    / weighted_order_discount["total_sales"],
    np.nan
)

order_discount_analysis = order_economics.merge(
    weighted_order_discount[
        ["order_id", "weighted_avg_discount"]
    ],
    on="order_id",
    how="left"
)

order_discount_analysis["discount_band"] = pd.cut(
    order_discount_analysis["weighted_avg_discount"],
    bins=[-0.001, 0.10, 0.20, 0.30, 1.00],
    labels=[
        "0–10%",
        ">10–20%",
        ">20–30%",
        ">30%"
    ]
)

order_discount_band = (
    order_discount_analysis.groupby(
        "discount_band",
        observed=True
    )
    .agg(
        orders=("order_id", "count"),
        sales=("order_sales", "sum"),
        profit=("order_profit", "sum")
    )
    .reset_index()
)

order_discount_band["profit_margin_pct"] = (
    order_discount_band["profit"]
    / order_discount_band["sales"]
    * 100
)

order_discount_band["loss_making_orders"] = (
    order_discount_analysis[
        order_discount_analysis["order_profit"] < 0
    ]
    .groupby("discount_band", observed=True)["order_id"]
    .count()
    .reindex(
        order_discount_band["discount_band"],
        fill_value=0
    )
    .values
)

order_discount_band["loss_order_rate_pct"] = (
    order_discount_band["loss_making_orders"]
    / order_discount_band["orders"]
    * 100
)

display(
    order_discount_band.round({
        "sales": 2,
        "profit": 2,
        "profit_margin_pct": 2,
        "loss_order_rate_pct": 2
    })
)

,discount_band,orders,sales,profit,profit_margin_pct,loss_making_orders,loss_order_rate_pct
0,0–10%,15931,8633728,2019879.00,23.40,300,1.88
1,>10–20%,2988,1742459,207572.87,11.91,609,20.38
2,>20–30%,1181,726402,-17247.31,-2.37,568,48.09
3,>30%,4935,1540316,-741169.73,-48.12,4647,94.16


### Order-Level Discount Exposure Finding

Order-level analysis reveals a strong relationship between sales-weighted discount levels and profitability.

Orders with discounts between 0% and 10% generated approximately £8.63 million in sales and £2.02 million in profit, achieving a 23.40% margin. Only 1.88% of orders in this group were loss-making.

As discount levels increased, profitability deteriorated substantially.

Orders discounted above 10% and up to 20% achieved an 11.91% margin, with 20.38% of orders generating losses.

Orders discounted above 20% and up to 30% produced a negative aggregate margin of -2.37%, while 48.09% of orders were loss-making.

The most severe performance occurred among orders with sales-weighted discounts above 30%. This group contained 4,935 orders, generated approximately £1.54 million in sales, and produced losses of approximately £741,170. Furthermore, 94.16% of orders in this group were loss-making.

These findings provide strong evidence that aggressive discounting is associated with significant commercial risk.

Management should investigate discount approval policies, product-level pricing, and the profitability of heavily discounted orders before introducing new discount restrictions.

The findings are descriptive and do not establish that discounting alone causes the observed losses.

## 31. Data Validation

In [39]:
print("DATA VALIDATION")
print("-" * 45)

print(f"Original transaction lines: {len(df):,}")
print(f"Order-level records: {len(order_economics):,}")
print(f"Unique order IDs: {df['order_id'].nunique():,}")

print("\nSALES RECONCILIATION")
print(
    f"Original sales: £{df['sales'].sum():,.2f}"
)
print(
    f"Order-level sales: £{order_economics['order_sales'].sum():,.2f}"
)

print("\nPROFIT RECONCILIATION")
print(
    f"Original profit: £{df['profit'].sum():,.2f}"
)
print(
    f"Order-level profit: £{order_economics['order_profit'].sum():,.2f}"
)

print("\nDISCOUNT BAND VALIDATION")
print(
    f"Orders in discount bands: "
    f"{order_discount_band['orders'].sum():,}"
)

print(
    f"Loss-making orders in discount bands: "
    f"{order_discount_band['loss_making_orders'].sum():,}"
)

print("\nFINAL CHECKS")

print(
    "Sales match:",
    np.isclose(
        df["sales"].sum(),
        order_economics["order_sales"].sum()
    )
)

print(
    "Profit matches:",
    np.isclose(
        df["profit"].sum(),
        order_economics["order_profit"].sum()
    )
)

print(
    "All orders included:",
    order_discount_band["orders"].sum()
    == len(order_economics)
)

print(
    "Loss-making orders match:",
    order_discount_band["loss_making_orders"].sum()
    == len(loss_orders)
)

DATA VALIDATION
---------------------------------------------
Original transaction lines: 51,290
Order-level records: 25,035
Unique order IDs: 25,035

SALES RECONCILIATION
Original sales: £12,642,905.00
Order-level sales: £12,642,905.00

PROFIT RECONCILIATION
Original profit: £1,469,034.82
Order-level profit: £1,469,034.82

DISCOUNT BAND VALIDATION
Orders in discount bands: 25,035
Loss-making orders in discount bands: 6,124

FINAL CHECKS
Sales match: True
Profit matches: True
All orders included: True
Loss-making orders match: True


### Final Data Validation

The completed analysis was validated against the original MegaStore dataset.

- Original transaction lines: 51,290
- Unique orders: 25,035
- Total sales: £12,642,905.00
- Total profit: £1,469,034.82
- Loss-making orders: 6,124

All four reconciliation checks passed.

The order-level sales and profit totals match the original transaction-level dataset. All 25,035 orders are represented in the discount analysis, and the number of loss-making orders reconciles correctly.

The Python analysis is complete and ready for SQL validation and business intelligence reporting.